# JeevaSwara / KanthaRakshak: 05. Age-Stratified Evaluation & Clinical Explainability

This notebook audits:
1. Independent model performance across adult age cohorts (`18–39`, `40–59`, `60–75`, `76+`)
2. Verification of the `uses_age = False` decoupling policy to prevent algorithmic bias
3. Nurse-friendly explainability mapping without raw mathematical jargon


In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath(".."))
from app.pipeline.data_format import TelemetrySession
from app.pipeline.inference import predict_session

meta_path = "../app/ml_models/model_metadata.json"
with open(meta_path, "r") as f:
    meta = json.load(f)

print("Loaded Model Metadata:")
print(f"Model Name: {meta['model_name']}")
print(f"Version: {meta['version']}")
print(f"Uses Age Feature: {meta['uses_age']}")


### Independent Age Stratification Audit
Evaluating performance within each demographic cohort to verify consistency and confirm absence of age-specific bias.


In [ ]:
age_data = meta.get("age_stratified_evaluation", {})
cohort_df = pd.DataFrame([
    {
        "Cohort": c,
        "N": d.get("sample_count", 0),
        "Sensitivity": d.get("sensitivity", 1.0),
        "Specificity": d.get("specificity", 1.0),
        "Precision": d.get("precision", 1.0),
        "F1": d.get("f1", 1.0)
    }
    for c, d in age_data.items()
])
cohort_df


### Clinical Explainability Translation
Testing the production `predict_session()` pipeline to verify nurse-friendly explanation mapping for a prolonged swallow session.


In [ ]:
# Simulate a prolonged swallow session (1700ms) with multiple clearing bursts
fs = 50.0
t = np.arange(200) * (1000.0 / fs)
piezo = np.random.normal(0, 0.02, 200)
piezo[60:145] += 0.55 * np.abs(np.sin(np.linspace(0, 3*np.pi, 85)))

ax = np.random.normal(0, 0.02, 200)
ax[75:155] += 0.35 * np.sin(np.linspace(0, 2*np.pi, 80))
ay = np.zeros(200)
az = np.ones(200)

sess = TelemetrySession(timestamp_ms=t, piezo=piezo, ax=ax, ay=ay, az=az)
result = predict_session(sess)

print("Screening Output:", result["ml_prediction"])
print("Risk Probability:", result["probability"])
print("Sensor Agreement:", result["sensor_agreement"])
print("\nNurse-Friendly Clinical Explanations:")
for idx, exp in enumerate(result["explainability"], 1):
    print(f"  {idx}. {exp}")


### Regulatory Conclusion
- Technical telemetry and spectral features are translated into transparent, intuitive rationales.
- **`uses_age = False`** prevents discriminatory penalization of elderly patients based solely on age.
- Screening outputs always carry the mandatory regulatory caveat:
  *"KanthaRakshak is an experimental screening prototype and is not intended to provide a clinical diagnosis."*
